# Módulo práctico — Las **reglas** del OWASP CRS (SQLi · XSS · RCE · Path Traversal · CSRF) y cómo escribir las tuyas

**Asignatura:** SERRC / UPM · **Formato:** notebook de celdas **bash**, ~2 h, nivel máster. Espejo práctico de la baraja de clase, centrado en las **reglas** del Core Rule Set.

Este notebook corre **dentro del contenedor `waf-jupyter`**, en la red Docker `waf-net`. No usa variables: **cada petición lleva la URL y el payload completos y literales**, para que puedas hacer dos cosas con la misma línea — pegarla en el **navegador** de tu Mac, o lanzarla desde una **celda** con `curl`.

| Qué | Desde una celda (dentro de Docker) | Desde tu navegador (en tu Mac) |
|---|---|---|
| Juice Shop **con WAF** | `http://waf:80/rest/products/search?q=limon` | `http://localhost:8080/rest/products/search?q=limon` |
| Juice Shop **sin WAF** (backend directo) | `http://juice-shop:3000/rest/products/search?q=limon` | `http://localhost:3000/rest/products/search?q=limon` |
| Reglas y recarga | `docker exec waf-proxy nginx -s reload` | — |
| Log de auditoría (JSON) | `/workspace/waf-audit-log/audit.log` | — |

> **Nota sobre espacios en las URLs.** Donde un payload lleva espacios, los escribimos como **`%20`** (su forma codificada): así la línea funciona tanto pegada en el navegador como lanzada con `curl` (curl rechaza una URL con espacios en crudo y devolvería `HTTP 000`). En la barra del navegador también puedes teclear los espacios tal cual — el navegador los codifica solo.

> **Importante sobre los números.** Este laboratorio usa **OWASP CRS 4.0.0**. Los IDs de regla, las severidades y las puntuaciones de *anomaly score* **se leen del log real** en el momento de ejecutar: no hay ni un número escrito a mano. Donde la teoría cita un dato concreto va marcado como *«según la versión de CRS; compruébalo en el log»*.


El módulo tiene **dos partes**: la **Parte 1** dispara ataques y **lee** qué reglas saltan; la **Parte 2** te hace **escribir tus propias reglas** en `waf/tuning-local.conf`, cargarlas en caliente y verlas actuar.


In [ ]:
echo "Este notebook corre dentro del contenedor: $(hostname)"
echo
echo "Los tres contenedores del laboratorio:"
docker ps --format "table {{.Names}}\t{{.Image}}\t{{.Status}}"
echo
echo "Resolucion por nombre en la red waf-net:"
getent hosts juice-shop
getent hosts waf
echo
echo "Log de auditoria (lo leeremos en cada ataque):"
ls -l /workspace/waf-audit-log/audit.log
echo
echo "Orden de carga de la config del WAF (lo ultimo gana) — tuning-local.conf es el ultimo:"
docker exec waf-proxy cat /etc/nginx/modsec/main.conf | grep -E '^Include'


### (Opcional) Empezar con el log de auditoría en blanco

El log (`/workspace/waf-audit-log/audit.log`) **vive en el host** (carpeta `waf-audit-log/`), así que **no** se borra al reiniciar el contenedor: `docker compose restart`, `up` o `down` lo **conservan**. Para quedarte solo con los registros **de esta sesión**, vacíalo con `truncate` (el WAF sigue escribiendo desde cero). Ejecuta esta celda una vez, al empezar la práctica.


In [ ]:
echo "Vaciamos el log de auditoria para empezar la practica en blanco:"
: > /workspace/waf-audit-log/audit.log
echo "  audit.log -> $(wc -l < /workspace/waf-audit-log/audit.log) lineas"
echo "  (si tras un ataque no vieras entradas nuevas, reabre el log con:  docker exec waf-proxy nginx -s reload)"


---
# PARTE 1 — Disparar y **leer** las alertas del CRS, ataque por ataque

Para cada familia de ataque: una **petición en claro** (payload entero dentro de la URL, sin variables) que puedes pegar en el navegador **o** lanzar desde la celda, y justo después una celda que **lee del log real** qué reglas dispararon y con qué *anomaly score*. La tabla sale siempre del log: `id de regla | severidad | mensaje`.


## 1.1 SQL Injection — reglas `942xxx`

> *Baraja, §SQLi:* cuando el input se **concatena** en la consulta sin parametrizar, el atacante altera la estructura lógica. `admin' OR '1'='1` hace la condición siempre verdadera y el `WHERE` deja de filtrar. CRS lo detecta con `REQUEST-942-APPLICATION-ATTACK-SQLI.conf`: la **942100** usa `@detectSQLi` (**libinjection**, análisis léxico, no una firma de texto), más reglas para tautologías, *UNION*, bypass de autenticación, etc.

Payload en claro (tautología): **`' OR 1=1--`**  (los `%20` de la URL son los espacios)

<details><summary><b>Comprueba que lo has entendido:</b> ¿por qué 942100 detecta variantes que nunca ha «visto» antes?</summary>

Porque no compara el texto contra una lista de payloads conocidos: con `@detectSQLi` hace un **análisis léxico** de la cadena y la clasifica según *fingerprints* de la gramática de SQL. Cualquier cadena que «se lee como SQL» encaja, aunque sea nueva. Es la diferencia entre detección semántica y detección por firma literal.
</details>


In [ ]:
echo "SQL Injection — payload en claro (tautologia):   ' OR 1=1--   (%20 = espacio)"
echo
echo "  NAVEGADOR (con WAF):     http://localhost:8080/rest/products/search?q='%20OR%201=1--"
echo "  NAVEGADOR (sin WAF):     http://localhost:3000/rest/products/search?q='%20OR%201=1--"
echo
echo "  Desde esta CELDA:"
curl -s -o /dev/null -w "    sin WAF (juice-shop:3000) -> HTTP %{http_code}  (la inyeccion llega a la app)\n" "http://juice-shop:3000/rest/products/search?q='%20OR%201=1--"
curl -s -o /dev/null -w "    con WAF (waf:80)          -> HTTP %{http_code}  (bloqueado por reglas 942xxx)\n" "http://waf:80/rest/products/search?q='%20OR%201=1--"
sleep 1


In [ ]:
echo "Reglas que dispararon en la ULTIMA peticion bloqueada (leido del log real):"
echo
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId!="949110") | [.details.ruleId,.details.severity,.message] | @tsv' \
 | awk -F'\t' 'BEGIN{n[2]="CRITICAL";n[3]="ERROR";n[4]="WARNING";n[5]="NOTICE";
     print "  REGLA   | SEVERIDAD      | MENSAJE";
     print "  --------+----------------+-------------------------------------------"}
   {s=$2; sn=(s in n)?n[s]:"?"; printf "  %-7s | %-14s | %s\n",$1,sn"("s")",substr($3,1,50)}'
echo
echo "Anomaly score total y decision final — lo escribe la regla 949110 (del log):"
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId=="949110") | "  " + .message'


### Verlo en el navegador: la inyección devolviendo **datos**

Con `curl` ves solo el código HTTP. En el **navegador** (o en la celda de abajo) ves **el contenido** que saca la inyección. El buscador de Juice Shop es inyectable, así que **sin WAF** la consulta manipulada devuelve datos que no deberías ver:

- **Sin WAF — romper el filtro y listar de más:** `http://localhost:3000/rest/products/search?q=qwert'))--`
  La `'))--` cierra y comenta la consulta: el `WHERE` deja de filtrar y la API devuelve productos que la búsqueda normal no daría.
- **Sin WAF — volcar la tabla de usuarios (UNION):** la consulta hace `SELECT * FROM Products`, así que el `UNION` tiene que llevar **tantas columnas como tiene la tabla `Products`**. En Juice Shop **recientes son 10** (`id, name, description, price, deluxePrice, image, alternateImages, createdAt, updatedAt, deletedAt`); en versiones algo más antiguas son **9** (sin `alternateImages`). Para **10 columnas**:
  `http://localhost:3000/rest/products/search?q=qwert')) UNION SELECT id,email,password,'4','5','6','7','8','9','10' FROM Users--`
  El JSON (y la tienda) mostrará los **correos** y **hashes** de `Users` en el sitio del nombre y la descripción.

  > **Si te sale `500 Error: SQLITE_ERROR: SELECTs to the left and right of UNION do not have the same number of result columns`:** ¡eso ya es un **SQLi con éxito** (error-based: la app te confirma que inyectas y que usa SQLite)! Solo falta **cuadrar el nº de columnas**: cuenta los campos de un producto en una búsqueda normal (`http://localhost:3000/rest/products/search?q=apple`) o prueba primero con 9 y luego con 10, hasta que el error desaparezca y veas los datos.

- **Con WAF:** la misma petición contra `http://localhost:8080/rest/products/search?q=qwert'))--` → **403**: el WAF corta la inyección antes de la base de datos.

La celda de abajo te enseña el **cuerpo** de la respuesta sin WAF (lo que verías en el navegador) y el 403 con WAF.


In [ ]:
echo "SIN WAF: la inyeccion 'qwert'))--' rompe el filtro; inicio del cuerpo que devuelve la API:"
echo "--------------------------------------------------------------------------------"
curl -s "http://juice-shop:3000/rest/products/search?q=qwert'))--" | head -c 400; echo
echo "--------------------------------------------------------------------------------"
echo "(En el navegador veras ese mismo JSON; con el UNION correcto veras ademas correos/hashes.)"
echo
echo "CON WAF: la misma peticion se corta antes de llegar a la base de datos:"
curl -s -o /dev/null -w "   via WAF -> HTTP %{http_code}  (403: reglas 942xxx)\n" "http://waf:80/rest/products/search?q=qwert'))--"


### Cómo leer el log de auditoría del contenedor

El WAF escribe cada transacción relevante en un fichero de log, como cualquier servidor Linux. En este laboratorio ese fichero vive, **dentro del contenedor**, en la ruta absoluta `/workspace/waf-audit-log/audit.log`.

Cada línea es **una transacción** en JSON. Igual que en una máquina Linux real, lo lees con `tail`:

- **Las últimas N entradas:** `tail -n 5 /workspace/waf-audit-log/audit.log`
- **Seguirlo en vivo** mientras atacas desde el navegador (como harías con `tail -f /var/log/nginx/access.log` en un servidor; en Jupyter se para con el botón ■): `tail -f /workspace/waf-audit-log/audit.log`

En crudo es ilegible (una línea JSON enorme por transacción). El comando de la celda de abajo es el que conviene ejecutar para verlas **todas y ordenadas**: una fila por petición con hora, método, ruta, código HTTP y las reglas que dispararon.


In [ ]:
echo "Todas las transacciones del log, en orden:  hora | metodo ruta | codigo | reglas"
echo "--------------------------------------------------------------------------------"
jq -r '.transaction | .time_stamp + "  " + .request.method + " " + (.request.uri|.[0:55]) + "  -> HTTP " + (.response.http_code|tostring) + "  reglas: " + ([.messages[]?.details.ruleId]|join(","))' /workspace/waf-audit-log/audit.log
echo "--------------------------------------------------------------------------------"
echo "En vivo: tail -f /workspace/waf-audit-log/audit.log      Ultimas 5: tail -n 5 /workspace/waf-audit-log/audit.log"


## 1.2 Cross-Site Scripting (reflected) — reglas `941xxx`

> *Baraja, §XSS:* la app refleja input en el HTML **sin escapar** y el navegador lo ejecuta. **Reflected** viaja en la petición (el WAF lo ve); **stored** persiste; **DOM-based** vive en el JS de cliente y, si va en el fragmento `#`, el WAF **no puede verlo**. CRS: `REQUEST-941-APPLICATION-ATTACK-XSS.conf` con **941100** (`@detectXSS`, libinjection), **941110** (etiqueta `<script>`), **941160** (manejadores de evento) y **941320** (etiquetas HTML peligrosas).

Payload en claro: **`<script>alert(1)</script>`**

<details><summary><b>Comprueba que lo has entendido:</b> ¿por qué el mismo payload en el fragmento <code>#</code> de la URL no lo vería el WAF?</summary>

Porque el **fragmento** (todo lo que va tras `#`) **no se envía al servidor**: lo usa solo el navegador. Ni la app ni el WAF delante de ella llegan a verlo. Por eso el DOM-based XSS por fragmento queda, por diseño del protocolo, fuera del alcance de un WAF; se mitiga en el cliente.
</details>


In [ ]:
echo "XSS reflected — payload en claro:   <script>alert(1)</script>"
echo
echo "  NAVEGADOR (con WAF):     http://localhost:8080/rest/products/search?q=<script>alert(1)</script>"
echo "  NAVEGADOR (sin WAF):     http://localhost:3000/rest/products/search?q=<script>alert(1)</script>"
echo
echo "  Desde esta CELDA:"
curl -s -o /dev/null -w "    sin WAF (juice-shop:3000) -> HTTP %{http_code}\n" "http://juice-shop:3000/rest/products/search?q=<script>alert(1)</script>"
curl -s -o /dev/null -w "    con WAF (waf:80)          -> HTTP %{http_code}  (bloqueado por reglas 941xxx)\n" "http://waf:80/rest/products/search?q=<script>alert(1)</script>"
sleep 1


In [ ]:
echo "Reglas que dispararon en la ULTIMA peticion bloqueada (leido del log real):"
echo
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId!="949110") | [.details.ruleId,.details.severity,.message] | @tsv' \
 | awk -F'\t' 'BEGIN{n[2]="CRITICAL";n[3]="ERROR";n[4]="WARNING";n[5]="NOTICE";
     print "  REGLA   | SEVERIDAD      | MENSAJE";
     print "  --------+----------------+-------------------------------------------"}
   {s=$2; sn=(s in n)?n[s]:"?"; printf "  %-7s | %-14s | %s\n",$1,sn"("s")",substr($3,1,50)}'
echo
echo "Fijate: el XSS dispara VARIAS reglas 941xxx a la vez. En 1.5 veremos como SUMAN."
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId=="949110") | "  949110: " + .message'


### Verlo en el NAVEGADOR: un XSS que imprime "hola mundo"

Con `curl` contra la API solo vemos el **código HTTP**. Para *ver* un XSS ejecutarse hace falta un **navegador**: el payload se incrusta en la página y el navegador lo corre.

El buscador de Juice Shop refleja el término en la página. Pega esta URL en la barra de direcciones — el payload lanza una alerta con el texto **hola mundo**:

- **Sin WAF (backend directo):** `http://localhost:3000/#/search?q=<iframe src="javascript:alert('hola mundo')">`
- **Con WAF:** `http://localhost:8080/#/search?q=<iframe src="javascript:alert('hola mundo')">`

En los dos casos **salta la alerta "hola mundo"**. ¿Por qué el WAF no lo bloquea? Porque el payload va detrás de la almohadilla `#` (el **fragmento** de la URL), y el fragmento **no se envía al servidor**: lo usa solo el navegador. El WAF nunca lo ve → es un XSS **basado en DOM**, fuera de su alcance.

La celda de abajo lo confirma desde el otro lado: si ese mismo "hola mundo" va en la **query real** (`?q=`, que sí viaja al servidor), el WAF **sí** lo corta con 403.


In [ ]:
echo "El mismo 'hola mundo' pero en la QUERY real (?q=, que SI llega al servidor):"
curl -s -o /dev/null -w "  sin WAF (juice-shop:3000) -> HTTP %{http_code}  (la app recibe el payload)\n" "http://juice-shop:3000/rest/products/search?q=<script>alert('hola%20mundo')</script>"
curl -s -o /dev/null -w "  con WAF (waf:80)          -> HTTP %{http_code}  (403: reglas 941xxx lo cortan)\n" "http://waf:80/rest/products/search?q=<script>alert('hola%20mundo')</script>"
echo
echo "Y si va en el FRAGMENTO (tras '#'), curl ni lo envia -> el WAF no ve nada:"
curl -s -o /dev/null -w "  con WAF (fragmento)       -> HTTP %{http_code}  (no es 403: el fragmento no viaja)\n" "http://waf:80/#/search?q=hola-mundo"
echo "  -> por eso el XSS 'hola mundo' del navegador (URLs de arriba) se ejecuta aunque haya WAF."


## 1.3 Command Injection — reglas `932xxx`

> *Baraja, §OWASP A05 Injection:* la app construye un comando de shell con input del usuario; el `;` termina el comando legítimo y encadena otro (`127.0.0.1; cat /etc/passwd`). CRS lo detecta con `REQUEST-932-APPLICATION-ATTACK-RCE.conf`, que reconoce comandos Unix y metacaracteres de shell. Aunque el endpoint de Juice Shop no ejecute el comando, el WAF detecta el **patrón** en el perímetro.

Payload en claro (encadenamiento con `;`): **`127.0.0.1; cat /etc/passwd`**  (los `%20` de la URL son los espacios)

<details><summary><b>Comprueba que lo has entendido:</b> si el backend ni siquiera ejecuta ese comando, ¿qué sentido tiene que el WAF lo bloquee?</summary>

El WAF es una capa de **defensa en profundidad**: no sabe (ni debe asumir) si la app es vulnerable: corta el patrón malicioso en el perímetro por si lo fuera, y además te da **señal temprana** de que alguien está sondeando RCE. Bloquea en base a la forma de la petición, no al comportamiento del backend.
</details>


**El objetivo del ataque.** `cat /etc/passwd` intenta leer el fichero de usuarios de un servidor Linux. Antes de atacar, lo vemos con `cat` desde **este mismo contenedor**: el notebook corre dentro de `waf-jupyter`, que es una máquina Linux, así que `cat /etc/passwd` funciona aquí y te enseña el tipo de fichero que el atacante quiere sacar del servidor. Puedes verlo igual abriendo una **Terminal** en Jupyter (menú *File → New → Terminal*) y escribiendo el mismo comando.


In [ ]:
echo "Contenido de /etc/passwd de ESTE contenedor (waf-jupyter) — una maquina Linux:"
echo "--------------------------------------------------------------------------------"
# 'cat' es un comando estandar y esta en el contenedor.
cat /etc/passwd | head -n 10
echo "--------------------------------------------------------------------------------"
echo "Lo mismo en una Terminal de Jupyter (File > New > Terminal):  cat /etc/passwd"
echo
echo "Eso es lo que un ataque de LFI (lectura de ficheros) o de RCE (ejecucion de comandos)"
echo "querria robar del servidor. En las celdas siguientes lo intentamos via la API: el WAF"
echo "corta el intento con 403, asi que ese contenido nunca se filtra."


In [ ]:
echo "Command Injection — payload en claro (encadenamiento):   127.0.0.1; cat /etc/passwd   (%20 = espacio)"
echo
echo "  NAVEGADOR (con WAF):     http://localhost:8080/rest/products/search?q=127.0.0.1;%20cat%20/etc/passwd"
echo "  NAVEGADOR (sin WAF):     http://localhost:3000/rest/products/search?q=127.0.0.1;%20cat%20/etc/passwd"
echo
echo "  Desde esta CELDA:"
curl -s -o /dev/null -w "    sin WAF (juice-shop:3000) -> HTTP %{http_code}\n" "http://juice-shop:3000/rest/products/search?q=127.0.0.1;%20cat%20/etc/passwd"
curl -s -o /dev/null -w "    con WAF (waf:80)          -> HTTP %{http_code}  (bloqueado por reglas 932xxx)\n" "http://waf:80/rest/products/search?q=127.0.0.1;%20cat%20/etc/passwd"
sleep 1


In [ ]:
echo "Reglas que dispararon en la ULTIMA peticion bloqueada (leido del log real):"
echo
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId!="949110") | [.details.ruleId,.details.severity,.message] | @tsv' \
 | awk -F'\t' 'BEGIN{n[2]="CRITICAL";n[3]="ERROR";n[4]="WARNING";n[5]="NOTICE";
     print "  REGLA   | SEVERIDAD      | MENSAJE";
     print "  --------+----------------+-------------------------------------------"}
   {s=$2; sn=(s in n)?n[s]:"?"; printf "  %-7s | %-14s | %s\n",$1,sn"("s")",substr($3,1,50)}'
echo
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId=="949110") | "  949110: " + .message'


### Verlo en el navegador

- **Sin WAF:** `http://localhost:3000/rest/products/search?q=127.0.0.1;%20cat%20/etc/passwd`
- **Con WAF:** `http://localhost:8080/rest/products/search?q=127.0.0.1;%20cat%20/etc/passwd` → **403** (reglas 932xxx)

Aquí el navegador **no** te mostrará `/etc/passwd`: el buscador de Juice Shop **no ejecuta** comandos de shell, así que sin WAF la API responde una búsqueda normal (`success`, sin resultados para ese texto) — **no hay RCE real** en este punto. El **contenido que un RCE exitoso devolvería** es el `/etc/passwd` que ya viste con `cat` arriba (§1.3; puedes volver a mirarlo en la Terminal de Jupyter). El valor del WAF aquí es cortar el **patrón** de RCE en el perímetro aunque la app no fuese vulnerable: defensa en profundidad. La celda de abajo lo hace evidente.


In [ ]:
echo "SIN WAF: la API responde una busqueda NORMAL (no ejecuta el comando); inicio del cuerpo:"
curl -s "http://juice-shop:3000/rest/products/search?q=127.0.0.1;%20cat%20/etc/passwd" | head -c 200; echo
echo "  -> fijate: responde 'success' pero SIN el contenido de /etc/passwd (no hay RCE real aqui)."
echo "     El contenido que un RCE buscaria es el /etc/passwd de la celda 'cat' de arriba."
echo
echo "CON WAF: la misma peticion se corta:"
curl -s -o /dev/null -w "   via WAF -> HTTP %{http_code}  (403: reglas 932xxx)\n" "http://waf:80/rest/products/search?q=127.0.0.1;%20cat%20/etc/passwd"


## 1.4 Path Traversal / LFI — reglas `930xxx`

> *Baraja, §Taxonomía CRS:* el rango **930xxx** cubre *LFI / Path Traversal*: intentos de salir del directorio servido (`../../../../etc/passwd`) para leer ficheros del sistema. CRS reconoce tanto la secuencia `../` como nombres de ficheros sensibles conocidos (`/etc/passwd`).

Payload en claro (sin espacios, pegable tal cual): **`../../../../etc/passwd`**

Lo metemos **en el parámetro de búsqueda** (`?q=../../../../etc/passwd`) para que llegue a ModSecurity como un argumento y lo inspeccionen las reglas 930xxx. Hay un matiz de capas interesante: la **misma travesía escrita en la ruta** (`/..%2f..%2f..%2f..%2fetc%2fpasswd`) la rechaza **nginx** con un **400** *antes* de llegar a las reglas — nunca la ve el WAF. La celda muestra las dos cosas.

<details><summary><b>Comprueba que lo has entendido:</b> ¿por qué la travesía en la RUTA da 400 y en el PARÁMETRO da 403?</summary>

Son **capas distintas**. La ruta la procesa primero **nginx**, que considera `/..%2f..%2f..%2f..%2fetc%2fpasswd` una ruta malformada y responde **400 Bad Request** por su cuenta — la petición ni siquiera llega al motor de reglas. En cambio, el valor del parámetro `q` nginx lo pasa tal cual: ahí sí lo inspecciona **ModSecurity**, las reglas 930xxx detectan `../` y `/etc/passwd`, suman score y la 949110 bloquea con **403**. El 403 es «decisión del WAF»; el 400 es «decisión de nginx».
</details>


In [ ]:
echo "Path Traversal / LFI — payload en claro (sin espacios):   ../../../../etc/passwd"
echo
echo "  En el PARAMETRO de busqueda (lo inspecciona ModSecurity):"
echo "  NAVEGADOR (con WAF):     http://localhost:8080/rest/products/search?q=../../../../etc/passwd"
echo "  NAVEGADOR (sin WAF):     http://localhost:3000/rest/products/search?q=../../../../etc/passwd"
echo
echo "  Desde esta CELDA:"
curl -s -o /dev/null -w "    sin WAF (juice-shop:3000) -> HTTP %{http_code}\n" "http://juice-shop:3000/rest/products/search?q=../../../../etc/passwd"
curl -s -o /dev/null -w "    con WAF (waf:80)          -> HTTP %{http_code}  (bloqueado por reglas 930xxx)\n" "http://waf:80/rest/products/search?q=../../../../etc/passwd"
echo
echo "  Contraste de CAPAS: la misma travesia en la RUTA la rechaza nginx antes que las reglas:"
curl -s -o /dev/null -w "    ..%2f..%2f..%2fetc%2fpasswd en la RUTA via WAF -> HTTP %{http_code}  (400 = nginx rechaza la ruta malformada; no es el WAF)\n" "http://waf:80/..%2f..%2f..%2f..%2fetc%2fpasswd"
sleep 1


In [ ]:
echo "Reglas que dispararon en la ULTIMA peticion bloqueada con 403 (leido del log real):"
echo
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId!="949110") | [.details.ruleId,.details.severity,.message] | @tsv' \
 | awk -F'\t' 'BEGIN{n[2]="CRITICAL";n[3]="ERROR";n[4]="WARNING";n[5]="NOTICE";
     print "  REGLA   | SEVERIDAD      | MENSAJE";
     print "  --------+----------------+-------------------------------------------"}
   {s=$2; sn=(s in n)?n[s]:"?"; printf "  %-7s | %-14s | %s\n",$1,sn"("s")",substr($3,1,50)}'
echo
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId=="949110") | "  949110: " + .message'


### Verlo en el navegador

- **Sin WAF:** `http://localhost:3000/rest/products/search?q=../../../../etc/passwd`
- **Con WAF:** `http://localhost:8080/rest/products/search?q=../../../../etc/passwd` → **403** (reglas 930xxx)

Como en Command Injection, Juice Shop **no sirve** `/etc/passwd` desde `?q=`, así que por la API verás una respuesta normal, no el fichero. El **contenido que un Path Traversal / LFI exitoso devolvería** es ese mismo `/etc/passwd` de §1.3 (míralo con `cat` en la celda o en la Terminal de Jupyter). El WAF corta el patrón de travesía (`../` y `/etc/passwd`) en el perímetro. La celda de abajo lo hace evidente.


In [ ]:
echo "SIN WAF: la API responde una busqueda NORMAL (no lee el fichero); inicio del cuerpo:"
curl -s "http://juice-shop:3000/rest/products/search?q=../../../../etc/passwd" | head -c 200; echo
echo "  -> 'success' pero SIN /etc/passwd: Juice Shop no sirve ficheros por ?q=. El contenido que"
echo "     un Path Traversal exitoso devolveria es el /etc/passwd de la celda 'cat' de arriba."
echo
echo "CON WAF: la misma peticion se corta:"
curl -s -o /dev/null -w "   via WAF -> HTTP %{http_code}  (403: reglas 930xxx)\n" "http://waf:80/rest/products/search?q=../../../../etc/passwd"


## 1.5 Anomaly scoring — cómo **suman** los puntos y quién decide el 403

> *Baraja, §WAF:* CRS no bloquea al primer *match*. Cada regla que dispara **suma** puntos a `TX:ANOMALY_SCORE` según su severidad — **CRITICAL=5, ERROR=4, WARNING=3, NOTICE=2** —, y al final de la fase 2 la regla **949110** (`REQUEST-949-BLOCKING-EVALUATION.conf`) compara el total con el umbral (**5** por defecto *según la versión de CRS; compruébalo*). Si lo alcanza, bloquea con 403. Es un modelo **aditivo**: rara vez bloquea una sola regla, normalmente es la **suma** de varias señales.

Reusamos el XSS de 1.2 (dispara varias reglas 941xxx) y **leemos del log** cuánto suma cada una y el total real.

<details><summary><b>Comprueba que lo has entendido:</b> una petición dispara una sola regla de severidad WARNING. ¿La bloquea el WAF por defecto?</summary>

No. WARNING = **3 puntos**, y el umbral por defecto es **5**: `3 < 5`, así que la 949110 no bloquea (quedaría registrada pero pasa). Harían falta, por ejemplo, dos WARNING (3+3=6 ≥ 5) o una CRITICAL (5 ≥ 5). Comprueba el total y el umbral reales en el log, porque dependen de la versión del CRS.
</details>


In [ ]:
echo "Lanzamos un ataque que dispara VARIAS reglas (el XSS) para ver como suman:"
curl -s -o /dev/null "http://waf:80/rest/products/search?q=<script>alert(1)</script>"
sleep 1
echo
echo "Desglose leido del log real  (severidad -> puntos: CRITICAL=5, ERROR=4, WARNING=3, NOTICE=2):"
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId!="949110") | [.details.ruleId,.details.severity,.message] | @tsv' \
 | awk -F'\t' 'BEGIN{nm[2]="CRITICAL";pt[2]=5;nm[3]="ERROR";pt[3]=4;nm[4]="WARNING";pt[4]=3;nm[5]="NOTICE";pt[5]=2;
     printf "  %-8s %-13s %-4s %s\n","REGLA","SEVERIDAD","PTS","MENSAJE"}
   {s=$2; printf "  %-8s %-13s %-4s %s\n",$1,nm[s]"("s")",pt[s],substr($3,1,45); tot+=pt[s]}
   END{printf "\n  SUMA de puntos de las reglas disparadas = %d\n",tot}'
echo
echo "Lo que escribe el motor en la regla final 949110 (su 'Total Score' debe COINCIDIR con la suma):"
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId=="949110") | "  " + .message'
echo
echo "Umbral de anomaly score en crs-setup.conf (por defecto 5 — segun version de CRS, compruebalo):"
docker exec waf-proxy grep -nE 'inbound_anomaly_score_threshold' /etc/nginx/modsec/coreruleset/crs-setup.conf | sed 's/^/  /'
echo "  Si la linea aparece comentada con '#', rige el valor por defecto del CRS (5)."
echo "  Decision 949110: si  SUMA >= umbral  ->  403."


## 1.6 Paranoia Level — subirlo activa **más** reglas (y más falsos positivos)

> *Baraja, §WAF:* el **Paranoia Level** (1–4) decide **cuántas reglas** están activas, no el umbral. PL1 = solo patrones de alta confianza (pocos falsos positivos); subir el PL añade reglas más agresivas → más cobertura, más falsos positivos. Es un *trade-off* precision/recall que se ajusta por aplicación.

Lo demostramos con un input **legítimo**, `<h1>Hola</h1>`, que a **PL1 pasa (200)** y a **PL2 se convierte en falso positivo (403)**.

**Cómo se cambia el PL correctamente en este laboratorio** (y por qué `tuning-local.conf` *no* es el sitio para esto): el PL lo fija una `SecAction "id:900000,phase:1,pass,nolog,setvar:tx.blocking_paranoia_level=N"` (en `crs-setup.conf`) que debe cargarse **antes** que las reglas (`rules/*.conf`). Pero `tuning-local.conf` se carga **el último** (ver `main.conf`), así que una `SecAction` ahí llega **tarde** para el gating de las reglas de fase 1 que CRS ya evaluó. Y `crs-setup.conf` —el sitio natural— vive **dentro de la imagen** (no está montado desde el host), así que no se edita desde fuera. La vía correcta para la demo en caliente: editar `crs-setup.conf` **dentro del contenedor** con `docker exec` y recargar nginx. Lo dejamos en PL1 al final.

<details><summary><b>Comprueba que lo has entendido:</b> ¿por qué poner <code>setvar:tx.blocking_paranoia_level=2</code> en <code>tuning-local.conf</code> no sube el PL de forma fiable?</summary>

Porque `tuning-local.conf` se incluye **después** de `rules/*.conf`. Las reglas de CRS de **fase 1** que deciden si están activas según el PL ya se han evaluado cuando tu `SecAction` corre, así que para ellas llega tarde. Para que el PL rija **todas** las reglas, la variable debe fijarse **antes** de `rules/*.conf` — es decir, en `crs-setup.conf` (o en un fichero incluido antes de las reglas), no en el *tuning* final.
</details>


In [ ]:
echo "1) A PL1 (por defecto) una etiqueta HTML suelta pasa:"
echo "   NAVEGADOR: http://localhost:8080/rest/products/search?q=<h1>Hola</h1>"
echo "   CELDA:     http://waf:80/rest/products/search?q=<h1>Hola</h1>"
curl -s -o /dev/null -w "   via WAF a PL1 -> HTTP %{http_code}  (200 esperado: a PL1 no hay regla que la cace)\n" "http://waf:80/rest/products/search?q=<h1>Hola</h1>"
echo
echo "2) Subimos a PL2 por la VIA CORRECTA: en crs-setup.conf (se carga ANTES de rules/*.conf),"
echo "   editandolo DENTRO del contenedor (ese fichero vive en la imagen, no es editable desde el host):"
docker exec waf-proxy sed -i '/id:900001/d' /etc/nginx/modsec/coreruleset/crs-setup.conf
docker exec waf-proxy sh -c 'echo "SecAction \"id:900001,phase:1,nolog,pass,t:none,setvar:tx.blocking_paranoia_level=2,setvar:tx.detection_paranoia_level=2\"" >> /etc/nginx/modsec/coreruleset/crs-setup.conf'
docker exec waf-proxy nginx -t >/dev/null 2>&1 && docker exec waf-proxy nginx -s reload
sleep 1
curl -s -o /dev/null -w "   el MISMO input a PL2 -> HTTP %{http_code}  (ahora 403: falso positivo)\n" "http://waf:80/rest/products/search?q=<h1>Hola</h1>"
sleep 1
echo "   Regla(s) que se activan a PL2 y antes no (del log real):"
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId!="949110") | select(.message!="") | "     [" + .details.ruleId + "] " + .message'
echo
echo "3) Restauramos PL1 (quitamos la linea que anadimos) para dejar el lab como estaba:"
docker exec waf-proxy sed -i '/id:900001/d' /etc/nginx/modsec/coreruleset/crs-setup.conf
docker exec waf-proxy nginx -s reload
sleep 1
curl -s -o /dev/null -w "   comprobacion: <h1>Hola</h1> via WAF -> HTTP %{http_code}  (de nuevo 200)\n" "http://waf:80/rest/products/search?q=<h1>Hola</h1>"


## 1.7 CSRF — por qué CRS trae **0 reglas** para esto (y por qué eso motiva la Parte 2)

> *Baraja, §CSRF:* el navegador de una víctima autenticada ejecuta una petición no deseada; el servidor no la distingue de una acción real porque **la cookie de sesión viaja correctamente**. La SOP impide *leer* la respuesta cross-site, pero no *enviar* la petición. La defensa vive en la **aplicación** (token sincronizador, `SameSite`, validación de `Origin`/`Referer`), no en un patrón.

Clave para el WAF: una petición CSRF es **sintácticamente perfecta** — método, parámetros y cookie válidos. **No hay patrón que detectar**, así que CRS no trae reglas de CSRF y la petición forjada pasa. Esa es exactamente la motivación de la **Parte 2**: cuando el riesgo no tiene firma genérica, el defensor tiene que **escribir su propia regla**.

<details><summary><b>Comprueba que lo has entendido:</b> si CRS no cubre CSRF, ¿el WAF es inútil aquí?</summary>

No es que sea inútil: es que esta clase de ataque **no se resuelve por firma**. El WAF sigue cubriendo SQLi/XSS/RCE. Contra CSRF la defensa correcta es de la aplicación (token anti-CSRF, cookies `SameSite`). Lo que sí puede hacer un WAF es aplicar reglas **propias** de contexto (p. ej. exigir `Origin` en acciones sensibles) — y escribir reglas propias es justo lo de la Parte 2.
</details>


In [ ]:
echo "Cuantos ficheros de reglas de CRS mencionan 'csrf'? Los buscamos en los ficheros reales:"
docker exec waf-proxy sh -c "grep -rli csrf /etc/nginx/modsec/coreruleset/rules/ | wc -l" | xargs echo "  ficheros de reglas CRS con 'csrf':"
echo "  (0 = CRS no trae deteccion de CSRF: no hay patron generico que buscar)"
echo
echo "Peticion 'de accion' forjada con una cookie de sesion cualquiera, via WAF:"
curl -s -o /dev/null -w "  -> HTTP %{http_code}  (NO es 403: el WAF no ve ningun patron de ataque)\n" \
  -b "token=sesion-de-la-victima" "http://waf:80/rest/basket/1"
echo
echo "  El WAF no pone 403. Si ves un 401, lo pone la APLICACION (falta un JWT valido), no el WAF;"
echo "  con una cookie de sesion valida real, la accion se ejecutaria (200) y el WAF tampoco la"
echo "  frenaria. No hay regla 9xxxxx que distinga una accion legitima de una forjada."
echo
echo "Confirmacion: esa peticion no ha dejado una entrada de bloqueo (403) nueva del WAF."


---
# PARTE 2 — Crea **tu propia** regla y añádela al WAF

Las reglas propias van en **`waf/tuning-local.conf`**, que se carga **el último** (después de CRS), así que puede añadir o sobreescribir cualquier cosa. El fichero está **montado desde el host**, así que se edita y se aplica en caliente:

```
docker exec waf-proxy nginx -t          # comprobar sintaxis
docker exec waf-proxy nginx -s reload   # recargar sin reconstruir
```

**Convención de IDs.** El rango **900 000–999 999** es de OWASP CRS. El rango **1–99 999** está reservado para **reglas locales/propias** — y es el que usaremos (por ejemplo `1000`, `1001`, `1010`, `1020`). *(Enlaza con la diapositiva de numeración de reglas.)*

> **Detalle de montaje (importante).** `tuning-local.conf` está montado en el contenedor como **un único fichero**. Por eso estas celdas lo editan **sin cambiar su inodo**: reinician el fichero con `cat fichero.orig > tuning-local.conf` (truncar y reescribir en el sitio) y añaden reglas con `cat >> tuning-local.conf`. **Nunca** usan `sed -i` sobre él: `sed -i` crea un fichero nuevo (otro inodo) y **rompe el montaje de un solo fichero** (nginx diría «Not able to open file»). Si alguna vez te pasa, basta con `docker compose restart waf`.

Escribiremos tres reglas realistas. Cada celda de regla **parte siempre del `tuning-local.conf` limpio** y añade solo su regla, así que puedes ejecutarlas en cualquier orden; la última celda del módulo **restaura el fichero original**.


In [ ]:
echo "Copia de seguridad del tuning-local.conf ORIGINAL (se restaura al final del modulo):"
[ -f /workspace/waf/tuning-local.conf.orig ] || cp /workspace/waf/tuning-local.conf /workspace/waf/tuning-local.conf.orig
ls -l /workspace/waf/tuning-local.conf.orig
echo
echo "Rango de IDs reservado para reglas propias: 1 - 99999 (CRS usa 900000-999999)."
echo "Ciclo de trabajo para cada regla: editar tuning-local.conf  ->  nginx -t  ->  nginx -s reload."


## 2.A — Regla A: un marcador en claro **atado a un Paranoia Level** (`chain`)

Una `SecRule` que bloquea (403) cuando `ARGS:q` contiene el marcador **`ACTIVIDAD-SOSPECHOSA`**, **encadenada** (`chain`) con una comprobación `TX:BLOCKING_PARANOIA_LEVEL "@ge 2"`. Así se ata una regla a un PL: **a PL1 no actúa, a PL2 sí**.

Texto exacto de la regla (va en `tuning-local.conf`):

```
SecRule ARGS:q "@rx ACTIVIDAD-SOSPECHOSA" \
    "id:1001,phase:2,deny,status:403,t:none,log,msg:'Marcador sospechoso en q - regla propia 1001 (solo PL2)',tag:'regla-propia/parte2',chain"
    SecRule TX:BLOCKING_PARANOIA_LEVEL "@ge 2"
```

> **Ojo con el `msg`.** No metas **comas** dentro del valor de `msg:'texto del mensaje'`: el analizador de ModSecurity separa las acciones por comas, y una coma dentro del mensaje rompe la carga de la regla (`nginx -t` falla y la regla no se carga). Por eso aquí el mensaje usa un guion en vez de una coma.

> **Ojo con el orden del `chain`.** En libmodsecurity v3, una regla con `chain` debe ser **la última directiva de su bloque**. Si justo después pones otra directiva como el `SecAction` que fija el PL, el parser falla con *«Expecting an action, got: SecAction»*. Por eso, en la celda, el `SecAction id:1000` que sube el PL se carga **antes** de la Regla A, no después.

La dispara, en claro: **`http://localhost:8080/rest/products/search?q=ACTIVIDAD-SOSPECHOSA`** (navegador) · **`http://waf:80/rest/products/search?q=ACTIVIDAD-SOSPECHOSA`** (celda).

> **Nota sobre el PL aquí.** Para esta regla, que es de **fase 2**, basta con fijar `tx.blocking_paranoia_level=2` en `tuning-local.conf` (una `SecAction` de fase 1): cuando la 1001 se evalúa en fase 2, la variable ya vale 2. Para el *gating* global de las reglas de CRS en fase 1, no — ésa es la vía de la Parte 1 (`crs-setup.conf`).

<details><summary><b>Comprueba que lo has entendido:</b> ¿qué pasaría si quitaras el <code>chain</code> y la segunda <code>SecRule</code>?</summary>

La 1001 bloquearía **siempre** que apareciera `ACTIVIDAD-SOSPECHOSA`, con independencia del Paranoia Level. El `chain` la convierte en condicional: **ambas** condiciones (el marcador **y** PL≥2) deben cumplirse para que actúe. Es el mecanismo con el que una regla propia se «engancha» a un nivel de paranoia.
</details>


In [ ]:
# Partimos SIEMPRE del tuning-local.conf limpio (truncar en el sitio: NO cambia el inodo, no rompe el montaje)
[ -f /workspace/waf/tuning-local.conf.orig ] || cp /workspace/waf/tuning-local.conf /workspace/waf/tuning-local.conf.orig
cat /workspace/waf/tuning-local.conf.orig > /workspace/waf/tuning-local.conf

echo "== PL2: fijador de PL2 (id:1000) + Regla A =="
# OJO: el fijador SecAction va ANTES de la regla encadenada. En libmodsecurity v3
# una regla con 'chain' debe ser la ULTIMA directiva del bloque; si le sigue una
# SecAction, nginx -t falla con "Expecting an action, got: SecAction".
cat >> /workspace/waf/tuning-local.conf <<'EOF'

# ---- Fijador de Paranoia Level 2 (fase 1), ANTES de la regla encadenada ----
SecAction "id:1000,phase:1,pass,nolog,t:none,setvar:tx.blocking_paranoia_level=2"

# ---- REGLA A (practica): marcador en claro, solo actua a PL>=2 ----
SecRule ARGS:q "@rx ACTIVIDAD-SOSPECHOSA" \
    "id:1001,phase:2,deny,status:403,t:none,log,msg:'Marcador sospechoso en q - regla propia 1001 (solo PL2)',tag:'regla-propia/parte2',chain"
    SecRule TX:BLOCKING_PARANOIA_LEVEL "@ge 2"
EOF
docker exec waf-proxy nginx -t && docker exec waf-proxy nginx -s reload
sleep 1
echo "   http://localhost:8080/rest/products/search?q=ACTIVIDAD-SOSPECHOSA   (navegador)"
curl -s -o /dev/null -w "   via WAF -> HTTP %{http_code}  (403 esperado: la 1001 actua a PL2)\n" "http://waf:80/rest/products/search?q=ACTIVIDAD-SOSPECHOSA"
sleep 1
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId=="1001") | "   log: [1001] " + .message'

echo
echo "== PL1: la MISMA regla pero sin el fijador (reescribimos el fichero con solo la Regla A) =="
cat /workspace/waf/tuning-local.conf.orig > /workspace/waf/tuning-local.conf
cat >> /workspace/waf/tuning-local.conf <<'EOF'

# ---- REGLA A (practica): marcador en claro, solo actua a PL>=2 ----
SecRule ARGS:q "@rx ACTIVIDAD-SOSPECHOSA" \
    "id:1001,phase:2,deny,status:403,t:none,log,msg:'Marcador sospechoso en q - regla propia 1001 (solo PL2)',tag:'regla-propia/parte2',chain"
    SecRule TX:BLOCKING_PARANOIA_LEVEL "@ge 2"
EOF
docker exec waf-proxy nginx -t && docker exec waf-proxy nginx -s reload
sleep 1
curl -s -o /dev/null -w "   via WAF -> HTTP %{http_code}  (pasa: a PL1 la cadena 'PL>=2' es falsa, la 1001 no actua)\n" "http://waf:80/rest/products/search?q=ACTIVIDAD-SOSPECHOSA"


## 2.B — Regla B: umbral de peticiones por IP (*rate limiting* casero) → **429**

Contamos las peticiones de cada IP a una ruta y, superado un umbral (**50**) en una ventana de tiempo, devolvemos **429 Too Many Requests**. Se usa la **colección persistente por IP** de ModSecurity. Tres directivas, **en este orden**:

```
SecAction \
    "id:1010,phase:1,pass,nolog,initcol:ip=%{REMOTE_ADDR}"

SecRule REQUEST_URI "@beginsWith /rest/products/search" \
    "id:1011,phase:1,pass,nolog,setvar:ip.contador=+1,expirevar:ip.contador=60"

SecRule IP:contador "@gt 50" \
    "id:1012,phase:1,deny,status:429,log,severity:'WARNING',msg:'Umbral de peticiones superado (regla propia 1012)'"
```

- **`initcol:ip=%{REMOTE_ADDR}`** — abre (o recupera) la colección asociada a la IP del cliente: la «libreta» donde se apunta cuántas peticiones lleva esa IP.
- **`setvar:ip.contador=+1`** + **`expirevar:ip.contador=60`** — incrementa el contador de esa IP y le pone una caducidad de 60 s (ventana deslizante): si la IP para, el contador expira.
- **`SecRule IP:contador "@gt 50"` con acciones `deny,status:429,log,severity:'WARNING'`** — cuando el contador supera 50, corta con 429 y deja una alerta en el log.

La celda de la ráfaga lanza 55 peticiones seguidas: verás pasar las 50 primeras, el **429 a partir de la 51** y, en el log de alertas, **la regla 1012** que lo ha provocado.

<details><summary><b>Comprueba que lo has entendido:</b> ¿por qué el contador tiene que ir en una colección <code>ip</code> y no en una variable <code>tx</code>?</summary>

Porque `TX` es la colección de **la transacción en curso**: nace y muere con **una** petición, así que no puede «recordar» cuántas van. La colección `IP` **se mantiene entre peticiones** y está indexada por `REMOTE_ADDR`, así que acumula el conteo de esa IP a lo largo del tiempo. Sin persistencia no hay *rate limiting* posible.
</details>




In [ ]:
# Partimos del tuning-local.conf limpio (truncar en el sitio) y anadimos SOLO la Regla B
[ -f /workspace/waf/tuning-local.conf.orig ] || cp /workspace/waf/tuning-local.conf /workspace/waf/tuning-local.conf.orig
cat /workspace/waf/tuning-local.conf.orig > /workspace/waf/tuning-local.conf
cat >> /workspace/waf/tuning-local.conf <<'EOF'

# ---- REGLA B (practica): umbral de 50 peticiones por IP -> 429 ----
SecAction \
    "id:1010,phase:1,pass,nolog,initcol:ip=%{REMOTE_ADDR}"

SecRule REQUEST_URI "@beginsWith /rest/products/search" \
    "id:1011,phase:1,pass,nolog,setvar:ip.contador=+1,expirevar:ip.contador=60"

SecRule IP:contador "@gt 50" \
    "id:1012,phase:1,deny,status:429,log,severity:'WARNING',msg:'Umbral de peticiones superado (regla propia 1012)'"
EOF
docker exec waf-proxy nginx -t && docker exec waf-proxy nginx -s reload
sleep 1
echo
echo "Regla B cargada. La probamos en la celda siguiente con una rafaga de 55 peticiones."


In [ ]:
# Preparacion de la demo (no es parte de la regla): el contador se lleva por proceso de nginx,
# asi que durante la rafaga dejamos nginx con un solo proceso. Se restaura al final de la celda.
docker exec waf-proxy sed -ri 's/^[[:space:]]*worker_processes[^;]*;/worker_processes 1;/' /etc/nginx/nginx.conf
docker exec waf-proxy nginx -s reload >/dev/null 2>&1
sleep 2

echo "Lanzamos 55 peticiones seguidas a http://waf:80/rest/products/search?q=limon (umbral: 50)"
rm -f /tmp/rafaga.txt
for i in $(seq 1 55); do
  curl -s -o /dev/null -w "%{http_code}\n" "http://waf:80/rest/products/search?q=limon" >> /tmp/rafaga.txt
done
echo
echo "Resumen de codigos:"
sort /tmp/rafaga.txt | uniq -c | awk '{printf "  %s x HTTP %s\n",$1,$2}'
echo "  primer 429 en la peticion numero: $(grep -n '^429' /tmp/rafaga.txt | head -1 | cut -d: -f1)"
echo
echo "Alrededor del umbral (peticiones 48 a 55):"
nl -ba /tmp/rafaga.txt | sed -n '48,55p' | awk '{printf "  peticion %2d -> HTTP %s\n",$1,$2}'
sleep 1
echo
echo "Regla que ha saltado (leido del log real):"
echo
grep '"http_code":429' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction.messages[]? | [.details.ruleId,.details.severity,.message] | @tsv' \
 | awk -F'\t' 'BEGIN{n[2]="CRITICAL";n[3]="ERROR";n[4]="WARNING";n[5]="NOTICE";
     print "  REGLA   | SEVERIDAD      | MENSAJE";
     print "  --------+----------------+-------------------------------------------"}
   {s=$2; sn=(s in n)?n[s]:"?"; printf "  %-7s | %-14s | %s\n",$1,sn"("s")",substr($3,1,50)}'
echo
echo "La misma alerta en el log de auditoria (hora | metodo ruta | codigo | reglas):"
grep '"http_code":429' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/rest/products/search"))' | tail -1 \
 | jq -r '.transaction | "  " + .time_stamp + "  " + .request.method + " " + .request.uri + "  -> HTTP " + (.response.http_code|tostring) + "  reglas: " + ([.messages[]?.details.ruleId]|join(","))'

# Restauramos nginx a su configuracion normal
docker exec waf-proxy sed -ri 's/^[[:space:]]*worker_processes[^;]*;/worker_processes auto;/' /etc/nginx/nginx.conf
docker exec waf-proxy nginx -s reload >/dev/null 2>&1


## 2.C — Regla C: bloquear subida de ficheros con **extensión peligrosa** (`phase:2`, `FILES`)

Una `SecRule` sobre **`FILES`** (los nombres originales de los ficheros subidos en una petición *multipart*) que bloquea (403) cuando el nombre acaba en una extensión ejecutable/peligrosa. Va en **`phase:2`** (cuerpo de la petición).

```
SecRule FILES "@rx (?i)\.(php|phtml|jsp|asp|aspx|sh|exe|bat|cmd|dll|py|pl)$" \
    "id:1020,phase:2,deny,status:403,t:none,log,msg:'Subida de fichero con extension peligrosa: %{MATCHED_VAR}',tag:'regla-propia/parte2'"
```

> **Blacklist vs whitelist.** Esto es una **blacklist** (prohíbo lo que sé malo). Una **whitelist** de extensiones *permitidas* (p. ej. «solo `.pdf`, `.png`, `.txt`») sería **más segura**, porque la lista de prohibidas nunca termina (`.php5`, `.phar`, `.pht`, mayúsculas, dobles extensiones y muchas más). *(Enlaza con la diapositiva whitelist vs blacklist.)* La usamos como blacklist solo porque, como demostración de *«regla propia que inspecciona una subida»*, se ve más directa.

Dos formas de dispararla: **(1)** desde la celda con `curl -F` (creando antes un `.sh` inofensivo), contrastando con una subida legítima `.txt`; **(2)** en el **navegador**, con el formulario real de Juice Shop «Complain?» (`http://localhost:8080/#/complain`), que adjunta un fichero. Usamos `.sh` a propósito: extensiones como `.php` o `.jsp` **ya las bloquea el propio CRS** (reglas 933/944) antes de llegar a tu regla, así que con esas no verías actuar a la **tuya**; con `.sh` (que CRS no mira) el 403 y la entrada de log son de la **1020**. La regla corta en el **perímetro** aunque el endpoint no exista, porque inspecciona el nombre del fichero en fase 2 antes de reenviar al backend.

<details><summary><b>Comprueba que lo has entendido:</b> ¿por qué la regla va en <code>phase:2</code> y no en <code>phase:1</code>?</summary>

Porque el nombre del fichero viaja en el **cuerpo** *multipart* de la petición, y el cuerpo no está disponible hasta la **fase 2** (`REQUEST_BODY`). En fase 1 solo se ven la línea de petición y las cabeceras; la variable `FILES` aún no está poblada. Inspeccionar una subida es, por definición, trabajo de fase 2.
</details>


In [ ]:
# Partimos del tuning-local.conf limpio (truncar en el sitio) y anadimos SOLO la Regla C
[ -f /workspace/waf/tuning-local.conf.orig ] || cp /workspace/waf/tuning-local.conf /workspace/waf/tuning-local.conf.orig
cat /workspace/waf/tuning-local.conf.orig > /workspace/waf/tuning-local.conf
cat >> /workspace/waf/tuning-local.conf <<'EOF'

# ---- REGLA C (practica): bloquear subida con extension peligrosa (fase 2, FILES) ----
SecRule FILES "@rx (?i)\.(php|phtml|jsp|asp|aspx|sh|exe|bat|cmd|dll|py|pl)$" \
    "id:1020,phase:2,deny,status:403,t:none,log,msg:'Subida de fichero con extension peligrosa: %{MATCHED_VAR}',tag:'regla-propia/parte2'"
EOF
docker exec waf-proxy nginx -t && docker exec waf-proxy nginx -s reload
sleep 1
echo
echo "Regla C cargada. La probamos con una subida peligrosa y otra legitima en la celda siguiente."


In [ ]:
echo "Creamos un fichero .sh (peligroso) y uno .txt (legitimo), con contenido inofensivo:"
printf '%s\n' 'echo contenido inofensivo de laboratorio' > /tmp/prueba.sh
printf '%s\n' 'contenido de texto legitimo' > /tmp/prueba.txt
echo
echo "== Subida PELIGROSA (.sh) via WAF =="
echo "   (en el navegador: formulario real 'Complain?' en http://localhost:8080/#/complain)"
curl -s -o /dev/null -w "   curl -F 'file=@prueba.sh' -> HTTP %{http_code}  (403 = TU regla 1020 la corta en el perimetro)\n" \
  -F "file=@/tmp/prueba.sh" "http://waf:80/file-upload"
echo
echo "== Subida LEGITIMA (.txt) via WAF =="
curl -s -o /dev/null -w "   curl -F 'file=@prueba.txt' -> HTTP %{http_code}  (no la corta la 1020: extension no peligrosa)\n" \
  -F "file=@/tmp/prueba.txt" "http://waf:80/file-upload"
echo
echo "Entrada que deja la regla 1020 en el log real:"
sleep 1
grep '"http_code":403' /workspace/waf-audit-log/audit.log | jq -c 'select(.transaction.request.uri|contains("/file-upload"))' | tail -1 \
 | jq -r '.transaction.messages[]? | select(.details.ruleId=="1020") | "   [1020] " + .message'
echo
echo "Elegimos .sh a proposito: extensiones como .php o .jsp ya las bloquea el propio CRS"
echo "(reglas 933/944) antes de llegar a tu regla; TU 1020 amplia la cobertura a las que CRS no"
echo "mira, como .sh, .exe o .aspx. La 1020 inspecciona el nombre del fichero en fase 2 y corta"
echo "en el perimetro, antes del backend (el endpoint/campo exactos pueden variar segun la version)."


## 2.Z — Cierre y **restauración** del laboratorio

Dejamos `waf/tuning-local.conf` **tal y como estaba** (sin las reglas de práctica) y recargamos nginx. El Paranoia Level ya quedó en PL1 al final de 1.6, y los cambios de 1.6 sobre `crs-setup.conf` eran dentro del contenedor y ya se revirtieron. El laboratorio queda limpio.

**Lo que te llevas de este módulo.** La Parte 1 mostró que CRS **lee** la petición, **suma** señales por severidad y **decide** en la 949110 contra un umbral, y que subir el PL añade reglas (y falsos positivos) — todo leído del log real de CRS 4.0.0, sin números a mano. La Parte 2 mostró el otro lado: cuando el riesgo **no tiene firma genérica** (un marcador propio, un umbral de ritmo, una subida peligrosa, el CSRF), el defensor **escribe su propia regla** en el rango local `1–99 999`, la ata a un contexto (un Paranoia Level, una ruta, una fase) y la ve actuar.


In [ ]:
echo "Restauramos tuning-local.conf a su estado ORIGINAL (truncar en el sitio, sin cambiar el inodo):"
if [ -f /workspace/waf/tuning-local.conf.orig ]; then
  cat /workspace/waf/tuning-local.conf.orig > /workspace/waf/tuning-local.conf
  rm -f /workspace/waf/tuning-local.conf.orig
  echo "  restaurado desde la copia de seguridad."
else
  echo "  (no habia copia de seguridad: el fichero ya estaba en su estado original)."
fi
docker exec waf-proxy sed -ri 's/^[[:space:]]*worker_processes[^;]*;/worker_processes auto;/' /etc/nginx/nginx.conf
docker exec waf-proxy nginx -t && docker exec waf-proxy nginx -s reload
echo
echo "Comprobacion final: el marcador de practica ya NO se bloquea (regla 1001 retirada):"
curl -s -o /dev/null -w "  q=ACTIVIDAD-SOSPECHOSA via WAF -> HTTP %{http_code}  (ya no es 403)\n" "http://waf:80/rest/products/search?q=ACTIVIDAD-SOSPECHOSA"
echo
echo "Laboratorio limpio. Fin del modulo."
